## Connect to drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/vietlink_chatbot

/content/drive/MyDrive/vietlink_chatbot


In [3]:
!ls

 analysis_extraction_layer.ipynb   rag_clarifcation_layer.ipynb   results
 chroma_kb_store		  'Report 12 08 26.gdoc'	  test_samples


In [4]:
!pip install -q google-genai openai anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.5 MB/s eta 0:00:00


In [5]:
from __future__ import annotations

import argparse
import json
import os
import re
from pathlib import Path
from typing import Any, Callable, Dict, List

from google import genai
from google.genai import types as genai_types
from google.colab import userdata

import openai
import anthropic


In [6]:
SUPPORTED_EXTENSIONS = {".txt", ".md"}

NO_CASE_FOUND_MARKER = "No sufficiently clear ambiguous case found"

INPUT_DIR = Path('test_samples')
RESULTS_ROOT_DIR = Path('results')

PROVIDER_MODELS = {
    "gemini": {
        "prompt_1_model": "gemini-pro-latest",
        "prompt_2_model": "gemini-flash-lite-latest",
    },
    "gpt": {
        "prompt_1_model": "gpt-5",
        "prompt_2_model": "gpt-5-mini",
    },
    "claude": {
        "prompt_1_model": "claude-opus-4-8",
        "prompt_2_model": "claude-haiku-4-5-20251001",
    },
}


In [7]:
PROMPT_1_TEMPLATE = """\
You are an expert conversation analyst specializing in detecting \
AMBIGUITY, UNCLEAR COMMUNICATION, and MISSING CONTEXT.

Analyze the conversation below and identify every ambiguity that \
genuinely occurred.

RULES:
1. Ground every finding strictly in the conversation.
2. Do not invent missing context, intentions, or facts.
3. Infer speaker roles only from the conversation.
4. An ambiguity must have caused, or plausibly could have caused, \
   misunderstanding, misdirected advice, rework, or clarification.
5. Do NOT flag statements merely because they are brief or incomplete \
   if their meaning is clear from context.
6. Keep separate cases separate when their root causes differ.
7. If multiple cases have the same root cause, analyze them separately \
   but mark them for merging.
8. Prefer concise reasoning over repetition.

For EACH case, return:

### Case {{{{number}}}}

**Evidence:** \
Quote only the minimum relevant excerpts needed to prove the ambiguity.

**What happened:** \
1-3 sentences describing the misunderstanding or missing information.

**Root cause:** \
Explain WHY the ambiguity occurred, not merely what information was missing. \
Focus on the communication mechanism or assumption that caused it.

**Category:** \
Choose the most specific category. Examples:
Missing Context, Undefined Scope, Ambiguous Terminology, Missing Constraint,
Undefined Success Criteria, Conflicting Requirement, Missing Format/Data Spec,
Unclear Process/Flow, Ambiguous Reference, Undefined Audience/Recipient,
Framework/Tool Side Effect, Other.

**Context type:** \
Select one or more:
Task/Action, Information/Advice, Document/Communication,
Planning/Decision, Technical/Problem-Solving, Emotional/Personal, General.

**Severity:** High / Medium / Low
Briefly explain the consequence if the ambiguity were not clarified.

**Confidence:** High / Medium / Low
Briefly explain how directly the conversation supports the finding.

**Generalizable principle:** \
State ONE concise principle that can apply to similar future conversations.
If no reusable principle exists, write exactly:
NOT generalizable — exclude from KB.

If two or more cases share the same root cause, add:
**Merge note:** Case X and Case Y share the same root cause and should \
be merged into one knowledge entry.

If no sufficiently grounded ambiguity exists, respond exactly:
"{no_case_marker}."

<conversation>
{{conversation}}
</conversation>
""".format(no_case_marker=NO_CASE_FOUND_MARKER)

In [8]:
PROMPT_2_TEMPLATE = """\
You receive an analysis of ambiguous cases found in a conversation (root \
cause, category, priority, confidence, and generalizability have already \
been analyzed). Your task is NOT to re-analyze — it is to CONVERT this \
analysis into knowledge base entries following the exact JSON schema \
below.

Rules:

1. Skip every case marked "NOT generalizable — exclude from KB".

2. If the analysis notes that 2 cases should be merged (same root cause), \
create a SINGLE entry, merging both cases' evidence into the "source" \
array.

3. For each remaining entry, fill in:
   - knowledge_id: "KB-{{n}}" where n is the sequence number within this \
batch (a real, globally-unique ID is assigned later — this is only a \
placeholder).
   - title: a short name based on "What happened"/Category.
   - category: use the category suggested in the analysis, as-is \
(free-text, not required to match a fixed enum).
   - description: rewrite "Root cause analysis" into a general PRINCIPLE \
(drop case-specific details, keep the underlying issue).
   - intent: why applying this knowledge is useful — inferred from \
"Severity reasoning".
   - priority: use the value proposed in "Severity reasoning" as-is.
   - confidence: use the value proposed in "Confidence reasoning" as-is.
   - status: always "Draft".
   - applicable_to: from "Applicable context type".
   - trigger: a short condition to recognize when a NEW request should \
activate this entry (based on "What happened" + category).
   - detection_rules: more specific than trigger — the pattern/signal \
that confirms the issue genuinely exists in a request.
   - clarification_questions: specific questions, reusable verbatim when \
asking about the same kind of issue again.
   - recommendation: how to rewrite the request to avoid this ambiguity \
from the start.
   - expected_outcome: the desired state after applying the \
recommendation.
   - source: an array with 1 object (or more if cases were merged):
     - conversation_id: "{{conversation_id}}"
     - root_cause: taken as-is from "Root cause analysis"
     - evidence: taken verbatim from "Evidence" (do NOT paraphrase)
     - case_summary: from "What happened", shortened to 1-2 sentences

4. Do NOT add any information beyond what's already in the analysis. If a \
field lacks enough data to fill accurately, leave it as an empty string \
"" rather than making something up.

Output ONLY a JSON array following the schema below, no text outside the \
JSON.

Schema:
[
  {{
    "knowledge_id": "string",
    "title": "string",
    "category": "string",
    "description": "string",
    "intent": "string",
    "priority": "High | Medium | Low",
    "confidence": "High | Medium | Low",
    "status": "Draft",
    "applicable_to": ["Task/Action" | "Information/Advice" | "Document/Communication" | "Planning/Decision" | "Technical/Problem-Solving" | "Emotional/Personal" | "General"],
    "trigger": "string",
    "detection_rules": "string",
    "clarification_questions": ["string"],
    "recommendation": "string",
    "expected_outcome": "string",
    "source": [
      {{
        "conversation_id": "string",
        "root_cause": "string",
        "evidence": "string",
        "case_summary": "string"
      }}
    ]
  }}
]

<analysis_result>
{analysis_result}
</analysis_result>
"""

In [9]:
def call_gemini(
    prompt: str,
    model: str,
    temperature: float = 0,
    json_mode: bool = False,
    max_output_tokens: int = 8192,
) -> str:
    config = genai_types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_output_tokens,
        response_mime_type="application/json" if json_mode else None,
    )
    response = gemini_client.models.generate_content(model=model, contents=prompt, config=config)
    return response.text


def call_gpt(
    prompt: str,
    model: str,
    temperature: float = 0,
    json_mode: bool = False,
    max_output_tokens: int = 8192,
) -> str:

    response = openai_client.chat.completions.create(
        model=model,
        max_completion_tokens=max_output_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def call_claude(
    prompt: str,
    model: str,
    temperature: float = 0,
    json_mode: bool = False,
    max_output_tokens: int = 8192,
) -> str:

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=max_output_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in response.content if getattr(block, "type", "") == "text")


def _parse_json_array(raw_output: str) -> List[Dict[str, Any]]:

    cleaned = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", cleaned, re.DOTALL)
        data = json.loads(match.group(0)) if match else []
    return data if isinstance(data, list) else []


In [10]:
# ---------------------------------------------------------------------------
# Prompt 1 — Analysis
# ---------------------------------------------------------------------------

def run_analysis(call_llm_fn: Callable[..., str], model: str, conversation_text: str) -> str:
    prompt = PROMPT_1_TEMPLATE.format(conversation=conversation_text)
    return call_llm_fn(prompt, model, temperature=0.3, json_mode=False)


# ---------------------------------------------------------------------------
# Prompt 2 — Extraction
# ---------------------------------------------------------------------------

def run_extraction(call_llm_fn: Callable[..., str], model: str, analysis_result: str, conversation_id: str) -> List[Dict[str, Any]]:
    prompt = PROMPT_2_TEMPLATE.format(analysis_result=analysis_result, conversation_id=conversation_id)
    raw_output = call_llm_fn(prompt, model, temperature=0, json_mode=True)
    return _parse_json_array(raw_output)



def mine_kb_from_folder(
    call_llm_fn: Callable[..., str],
    prompt_1_model: str,
    prompt_2_model: str,
    input_dir: Path,
    output_dir: Path,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    reports_dir = output_dir / "analysis_reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    output_json_path = output_dir / "kbs.json"

    sample_files = sorted(
        f for f in input_dir.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    if not sample_files:
        raise FileNotFoundError(f"No .txt/.md sample files found in {input_dir}")

    all_entries: List[Dict[str, Any]] = []

    for sample_file in sample_files:
        conversation_id = sample_file.stem
        conversation_text = sample_file.read_text(encoding="utf-8")

        print(f"[{conversation_id}] Running Prompt 1 (Analysis)...")
        analysis_report = run_analysis(call_llm_fn, prompt_1_model, conversation_text)

        report_path = reports_dir / f"{conversation_id}.analysis.md"
        report_path.write_text(analysis_report, encoding="utf-8")
        print(f"[{conversation_id}] Analysis report saved -> {report_path}")

        if NO_CASE_FOUND_MARKER in analysis_report:
            print(f"[{conversation_id}] No ambiguous case found, skipping extraction.")
            continue

        print(f"[{conversation_id}] Running Prompt 2 (Extraction)...")
        entries = run_extraction(call_llm_fn, prompt_2_model, analysis_report, conversation_id)
        print(f"[{conversation_id}] Extracted {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}.")

        all_entries.extend(entries)

    for i, entry in enumerate(all_entries, start=1):
        entry["knowledge_id"] = f"KB-{i}"

    output_json_path.write_text(json.dumps(all_entries, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"\nDone. {len(all_entries)} KB entries written -> {output_json_path}")
    print(f"Per-file analysis reports saved under -> {reports_dir}")


In [11]:
gemini_client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
openai_client = openai.OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

CALL_FN_BY_PROVIDER: Dict[str, Callable[..., str]] = {
    #"gemini": call_gemini,
    #"gpt": call_gpt,
    "claude": call_claude,
}

for provider_name, models in PROVIDER_MODELS.items():
    if provider_name in CALL_FN_BY_PROVIDER.keys():
      print(f"\n===== Provider: {provider_name} =====")
      mine_kb_from_folder(
          call_llm_fn=CALL_FN_BY_PROVIDER[provider_name],
          prompt_1_model=models["prompt_1_model"],
          prompt_2_model=models["prompt_2_model"],
          input_dir=INPUT_DIR,
          output_dir=RESULTS_ROOT_DIR / provider_name,
      )



===== Provider: claude =====
[sample_1] Running Prompt 1 (Analysis)...
[sample_1] Analysis report saved -> results/claude/analysis_reports/sample_1.analysis.md
[sample_1] Running Prompt 2 (Extraction)...
[sample_1] Extracted 2 entries.
[sample_2] Running Prompt 1 (Analysis)...
[sample_2] Analysis report saved -> results/claude/analysis_reports/sample_2.analysis.md
[sample_2] Running Prompt 2 (Extraction)...
[sample_2] Extracted 6 entries.
[sample_3] Running Prompt 1 (Analysis)...
[sample_3] Analysis report saved -> results/claude/analysis_reports/sample_3.analysis.md
[sample_3] Running Prompt 2 (Extraction)...
[sample_3] Extracted 2 entries.
[sample_4] Running Prompt 1 (Analysis)...
[sample_4] Analysis report saved -> results/claude/analysis_reports/sample_4.analysis.md
[sample_4] Running Prompt 2 (Extraction)...
[sample_4] Extracted 5 entries.

Done. 15 KB entries written -> results/claude/kbs.json
Per-file analysis reports saved under -> results/claude/analysis_reports
